<a href="https://colab.research.google.com/github/ksuplee/AI_Agent/blob/main/09_2_MCP_%EA%B8%B0%EB%B0%98_%EC%BB%A8%ED%85%8D%EC%8A%A4%ED%8A%B8_%ED%86%B5%ED%95%A9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[실습 09-2] MCP(Model Context Protocol) 구조 기반 컨텍스트 통합 에이전트  

### 실습목표

- MCP Host-Server 아키텍처를 코드로 시뮬레이션하여 데이터와 도구가 모델에 주입되는 흐름을 이해한다 [cite: 09-2-1].  

- **Resource(읽기 전용 데이터)** 와 **Tool(실행 기능)** 의 차이점을 명확히 구분하고 에이전트에 통합한다 [cite: 09-2-2].  

- 도구 실행 결과가 새로운 컨텍스트로 순환되는 Closed-loop 구조를 구현한다 [cite: 09-2-3].  

1. 환경 준비 및 라이브러리 설치  

- MCP 개념을 구현하기 위해 LangChain 및 Google Gemini 설정을 진행합니다.  

In [ ]:
# 기존 설치를 무시하고 최신 버전으로 강제 재설치합니다.
!pip install -q -U --force-reinstall langchain langchain-community langchain-huggingface langchain-core langchain-google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.1/53.1 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.1/75.1 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.6/41.6 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.8/108.8 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 490.2/490.2 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 719.4/719.4 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1

In [ ]:
# Google API Key 설정
import google.generativeai as genai
from google.colab import userdata

GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

print("Gemini API 설정 완료")

Gemini API 설정 완료


In [ ]:
# 1. 필수 라이브러리 설치
# !pip install -q -U langchain langchain-community langchain-google-genai

import google.generativeai as genai
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain.tools import tool

# Gemini API 설정
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
llm = ChatGoogleGenerativeAI(model='gemini-flash-latest', api_key=GOOGLE_API_KEY)

2. MCP Server 시뮬레이션 (Resource & Tool 정의)  

- 교안에 명시된 MCP Server의 역할을 코드로 구현합니다. Server는 모델을 알 필요 없이 오직 **데이터(Resource)** 와 **기능(Tool)** 만 정의합니다 [cite: 09-2-1].  

In [ ]:
# --- [MCP Server 영역] --- [cite: 09-2-1]

# 1. Resources: 모델이 읽기 전용으로 참고하는 데이터 [cite: 09-2-2]
MCP_RESOURCES = {
    "user_manual": "본 에이전트는 복잡한 수학 계산과 시스템 로그 조회를 지원합니다. 계산 결과는 소수점 둘째 자리까지 출력하는 것이 원칙입니다.",
    "system_status": "현재 시스템 온도는 45도이며, 모든 데이터베이스 연결은 정상입니다."
}

# 2. Tools: 모델이 외부 세계에 영향을 미치는 실행 기능 [cite: 09-2-2]
@tool
def check_db_log(query: str) -> str:
    """데이터베이스 로그에서 특정 쿼리와 관련된 기록을 검색합니다."""
    # 시뮬레이션 실행 결과
    return f"로그 검색 결과: '{query}'와 관련된 에러 기록은 없습니다. 상태 양호."

mcp_tools = [check_db_log]

3. MCP Host 및 Client 구축 (Context Injection)  

- Host는 사용자 질문과 관련된 Resource를 선별하여 모델의 컨텍스트 윈도우에 직접 주입합니다 [cite: 09-2-1, 09-2-3].  

In [ ]:
# --- [MCP Host 영역] --- [cite: 09-2-1]

# 도구 바인딩
llm_with_tools = llm.bind_tools(mcp_tools)

# 1. Discovery & Context Injection [cite: 09-2-3]
def mcp_host_logic(user_input: str):
    # 질문과 관련된 리소스를 선택하여 컨텍스트에 주입 (시뮬레이션)
    # 실제 MCP 환경에서는 메타데이터 기반으로 자동 선별됨 [cite: 09-2-3]
    injected_context = f"관련 매뉴얼: {MCP_RESOURCES['user_manual']}\n시스템 상태: {MCP_RESOURCES['system_status']}"

    # 2. Prompt (가이드 템플릿) 적용 [cite: 09-2-2]
    prompt = ChatPromptTemplate.from_messages([
        ("system", "너는 MCP 환경에서 작동하는 에이전트야. 주입된 [Resource]를 근거로 답변하고 필요시 [Tool]을 사용해.\n\n[Resource]\n{context}"),
        ("human", "{input}")
    ])

    chain = prompt | llm_with_tools
    return chain.invoke({"context": injected_context, "input": user_input})

print("MCP Host-Server 시뮬레이션 준비 완료!")


--- [모델의 출력 결과 확인] ---
Tool Calls: [{'name': 'complex_calculator', 'args': {'expression': '12345 * 67890'}, 'id': '995112ed-3608-4e9a-8d92-2b73645d0f0a', 'type': 'tool_call'}]


4. 실습 테스트 (Integration & Closed-loop)  

- 도구 실행 결과가 새로운 리소스로 등록되어 다음 추론에 활용되는 흐름을 확인합니다 [cite: 09-2-3].  

In [ ]:
# --- [에이전트 실행 테스트] ---

query = "매뉴얼 원칙에 따라, 시스템 로그에서 'DB_CONN' 관련 기록을 확인해줘."
ai_msg = mcp_host_logic(query)

# 1. 모델이 Resource를 참고하여 Tool을 호출했는지 확인 [cite: 09-2-3]
if ai_msg.tool_calls:
    print(f"[*] 모델이 도구를 선택함: {ai_msg.tool_calls[0]['name']}")

    # 2. Execution: 도구 실행 [cite: 09-2-3]
    tool_call = ai_msg.tool_calls[0]
    observation = check_db_log.invoke(tool_call["args"])
    print(f"[*] 관찰 결과(Observation): {observation}")

    # 3. Integration: 실행 결과를 다시 컨텍스트로 활용 [cite: 09-2-3]
    final_answer = llm_with_tools.invoke([
        HumanMessage(content=query),
        ai_msg,
        ToolMessage(content=observation, tool_call_id=tool_call["id"])
    ])

    print("-" * 50)
    print(f"[최종 답변]:\n{final_answer.content}")

[*] 도구 실행 결과: 838102050


ChatGoogleGenerativeAIError: Error calling model 'gemini-flash-latest' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 37.694231313s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-Fr

- 관찰 포인트  
    - **데이터와 기능의 분리**: MCP Server는 모델에 종속되지 않고 데이터(Resource)와 함수(Tool)만 제공하는 구조적 독립성을 가집니다 [cite: 09-2-1].  
    - **Grounding 효과**: 모델은 자신이 학습하지 않은 system_status 리소스를 마치 처음부터 알고 있던 정보처럼 활용하여 답변합니다 [cite: 09-2-3].  
    - **표준화된 연결**: 이 코드는 모델을 변경하더라도(예: Anthropic Claude) 동일한 MCP_RESOURCES와 mcp_tools를 그대로 재사용할 수 있는 재사용성을 보장합니다 [cite: 09-2-4].  

**실습점검**  

- Q1. 실습 코드에서 '읽기 전용 매뉴얼'은 MCP의 어떤 요소에 해당하나요?  
    - A. Resources입니다. 모델의 컨텍스트 윈도우에 직접 주입되어 추론의 근거로 활용됩니다 [cite: 09-2-2].  

    
- Q2. Tool 실행 결과인 '관찰(Observation)'은 이후 대화에서 무엇이 되나요?  
    - A. 새로운 **Resource(컨텍스트)** 가 되어 다음 추론의 근거로 재활용됩니다 [cite: 09-2-3].  